In [ ]:
# Installer timm
!pip install timm

In [ ]:
import random
import os
import numpy as np
import torch

# Définition de la classe Config (avec SEED)
class Config:
    EPOCHS_PHASE1 = 2
    DATASET_FRACTION = 0.2
    IMG_SIZE = 224
    BATCH_SIZE = 32
    NUM_CLASSES = 2
    PREPROCESSED_DIR = "/content/drive/MyDrive/preprocessed_casia_balanced_verified"
    DATA_PATH = "/content/drive/MyDrive/CASIA2.0_revised_corrected/casia"
    MODEL_NAME = "late_fusion_tf_balanced_v2"
    EMBED_DIM = 256
    NUM_HEADS = 4
    NUM_LAYERS = 2
    DROPOUT = 0.2
    MEAN = [0.485, 0.456, 0.406]
    STD = [0.229, 0.224, 0.225]
    TRAIN_SIZE = 0.7
    VAL_SIZE = 0.15
    TEST_SIZE = 0.15
    SEED = 42

# Fonction pour fixer la seed
def set_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Utilisation
config = Config()
print("Seed dans Config:", config.SEED)
set_seed(config.SEED)
print("Seed utilisée et fixée.")


In [ ]:
# -*- coding: utf-8 -*-


import os
import torch
import numpy as np
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import timm
from PIL import Image, ImageFile
import logging
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import glob
from typing import Optional, Tuple, List, Dict
from torch.nn.modules.transformer import TransformerEncoderLayer
import torch.nn.functional as F
from sklearn.utils import shuffle
from torch.cuda.amp import GradScaler, autocast
import math # Needed for Positional Encoding calculation
import time # For timing epochs

# Allow loading of truncated images (use with caution)
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Configuration class
class Config:
    IMG_SIZE = 224
    BATCH_SIZE = 32 # Adjust based on GPU memory
    NUM_CLASSES = 2  # CASIA has 2 classes: authentic and tampered (will be verified)
    EPOCHS_PHASE1 = 5
    EPOCHS_PHASE2 = 10
    # --- IMPORTANT: VERIFY AND SET THESE PATHS ---
    # Path to the directory where PREPROCESSED .npz files WILL BE SAVED or LOADED FROM
    PREPROCESSED_DIR = "/content/drive/MyDrive/preprocessed_casia_balanced_verified"
    # Path to the ROOT directory containing the CASIA dataset (e.g., contains 'Au' and 'Tp' subdirs)
    DATA_PATH = "/content/drive/MyDrive/CASIA2.0_revised_corrected/casia"
    # --- END IMPORTANT PATHS ---

    DATASET_FRACTION = 1.0 # Use 1.0 for full dataset, smaller for testing
    LEARNING_RATE_PHASE1 = 1e-4
    LEARNING_RATE_PHASE2 = 1e-5
    EARLY_STOPPING_PATIENCE = 5
    CLIP_GRAD_NORM = 1.0
    SEED = 42
    # Note: Splits are applied *before* balancing the training set
    TRAIN_SIZE = 0.7
    VAL_SIZE = 0.15
    TEST_SIZE = 0.15
    MEAN = [0.485, 0.456, 0.406]  # ImageNet mean
    STD = [0.229, 0.224, 0.225]  # ImageNet std
    MODEL_NAME = "late_fusion_tf_balanced_v2" # Descriptive name for saved files
    # Transformer parameters
    EMBED_DIM = 256
    NUM_HEADS = 4
    NUM_LAYERS = 2
    DROPOUT = 0.2

config = Config()

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', datefmt='%Y-%m-%d %H:%M:%S')

# Set random seeds for reproducibility
def set_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    logging.info(f"Random seed set to {seed}")

set_seed(config.SEED)

# Device configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logging.info(f"Using device: {DEVICE}")

# --- Google Drive Mounting (with verification) ---
def mount_drive(mount_point='/content/drive'):
    try:
        from google.colab import drive
        logging.info(f"Attempting to mount Google Drive at {mount_point}...")
        drive.mount(mount_point, force_remount=True)
        # Verification step: Check if the mount point exists and is accessible
        if os.path.exists(mount_point) and os.path.isdir(mount_point) and os.access(mount_point, os.R_OK):
            logging.info(f"Google Drive mounted successfully and is accessible at {mount_point}.")
            # Optional: List top-level contents
            # logging.info(f"Contents of {mount_point}: {os.listdir(mount_point)[:10]}")
            return True
        else:
            logging.error(f"Mounting seemed to succeed, but {mount_point} is not accessible or is not a directory.")
            return False
    except ImportError:
        logging.warning("Not running in Google Colab environment. Skipping drive mount.")
        return False # Indicate mount didn't happen
    except Exception as e:
        logging.error(f"Error mounting Google Drive: {e}", exc_info=True)
        return False

IS_DRIVE_MOUNTED = mount_drive()

# --- Dataset Path Verification ---
def verify_dataset_paths(data_path):
    """Checks if the main data path and expected subdirectories exist."""
    logging.info(f"Verifying dataset structure at: {data_path}")
    if not data_path or not isinstance(data_path, str):
        logging.error("Config.DATA_PATH is not set or is not a string.")
        return False

    if not os.path.exists(data_path):
        logging.error(f"Dataset path does NOT exist: {data_path}")
        return False
    if not os.path.isdir(data_path):
        logging.error(f"Dataset path is NOT a directory: {data_path}")
        return False
    logging.info(f"Dataset path exists and is a directory: {data_path}")

    # Check for expected subdirectories 'Au' and 'Tp'
    authentic_dir = os.path.join(data_path, "Au")
    tampered_dir = os.path.join(data_path, "Tp")
    paths_ok = True

    if not os.path.exists(authentic_dir) or not os.path.isdir(authentic_dir):
        logging.error(f"Expected 'Au' subdirectory NOT found or is not a directory in: {data_path}")
        paths_ok = False
    else:
        logging.info(f"Found 'Au' subdirectory: {authentic_dir}")
        try:
            logging.info(f"  - Contains approx {len(os.listdir(authentic_dir))} items.")
        except OSError as e:
            logging.warning(f"Could not list contents of {authentic_dir}: {e}")


    if not os.path.exists(tampered_dir) or not os.path.isdir(tampered_dir):
        logging.error(f"Expected 'Tp' subdirectory NOT found or is not a directory in: {data_path}")
        paths_ok = False
    else:
        logging.info(f"Found 'Tp' subdirectory: {tampered_dir}")
        try:
             logging.info(f"  - Contains approx {len(os.listdir(tampered_dir))} items.")
        except OSError as e:
            logging.warning(f"Could not list contents of {tampered_dir}: {e}")

    if not paths_ok:
         logging.error("Dataset structure verification failed. Please check Config.DATA_PATH and the 'Au'/'Tp' subdirectories.")
    else:
         logging.info("Dataset structure verification successful ('Au' and 'Tp' found).")

    return paths_ok

# Perform path verification early
PATHS_VERIFIED = verify_dataset_paths(config.DATA_PATH)

# --- Dataset Class ---
class CASIADataset(Dataset):
    def __init__(self, images: np.ndarray, labels: np.ndarray, transforms: callable):
        if not isinstance(images, np.ndarray) or not isinstance(labels, np.ndarray):
             raise TypeError(f"Images and labels must be numpy arrays, got {type(images)} and {type(labels)}")
        if images.shape[0] != labels.shape[0]:
            raise ValueError(f"Number of images ({images.shape[0]}) and labels ({labels.shape[0]}) must match.")

        self.images = images
        self.labels = labels
        self.transforms = transforms
        self.dataset_len = len(self.images) # Store length for efficiency

    def __len__(self):
        return self.dataset_len

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        if idx < 0 or idx >= self.dataset_len:
            raise IndexError(f"Index {idx} out of bounds for dataset of size {self.dataset_len}")

        image_np = self.images[idx]
        label_val = self.labels[idx]

        # Basic check before converting to PIL
        if image_np.ndim != 3 or image_np.shape[2] != 3:
            logging.error(f"Image at index {idx} has unexpected shape {image_np.shape}. Returning dummy data.")
            # Return a dummy tensor and label to avoid crashing the loader
            dummy_image = torch.zeros((3, config.IMG_SIZE, config.IMG_SIZE), dtype=torch.float32)
            dummy_label = torch.tensor(-1, dtype=torch.long) # Indicate error with label -1
            return dummy_image, dummy_label

        # Ensure image is uint8 for PIL
        if image_np.dtype != np.uint8:
             try:
                 if image_np.min() >= 0 and image_np.max() <= 1.0: # Handle float 0-1
                     image_np = (image_np * 255).astype(np.uint8)
                 elif image_np.min() >= 0 and image_np.max() <= 255: # Handle float 0-255 or other int types
                     image_np = image_np.astype(np.uint8)
                 else:
                     logging.warning(f"Image at index {idx} has unexpected dtype {image_np.dtype} and range [{image_np.min()}, {image_np.max()}]. Attempting astype(uint8).")
                     image_np = image_np.astype(np.uint8)
             except (ValueError, TypeError) as e:
                 logging.error(f"Failed to convert image at index {idx} (dtype {image_np.dtype}) to uint8: {e}. Returning dummy data.")
                 dummy_image = torch.zeros((3, config.IMG_SIZE, config.IMG_SIZE), dtype=torch.float32)
                 dummy_label = torch.tensor(-1, dtype=torch.long)
                 return dummy_image, dummy_label

        try:
            image = Image.fromarray(image_np)
            image = self.transforms(image) # Apply augmentations/normalization
            label = torch.tensor(label_val, dtype=torch.long)
            return image, label
        except Exception as e:
            logging.error(f"Error applying transforms to image at index {idx}: {e}", exc_info=True)
            # Return dummy data on transformation error
            dummy_image = torch.zeros((3, config.IMG_SIZE, config.IMG_SIZE), dtype=torch.float32)
            dummy_label = torch.tensor(-1, dtype=torch.long)
            return dummy_image, dummy_label


# --- Data Loading and Preprocessing Function ---
def load_or_preprocess_data(data_path: str, preprocessed_dir: str, img_size: int, dataset_fraction: float = 1.0, train_split: float = 0.7, val_split: float = 0.15, test_split: float = 0.15, seed: int = 42) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Loads preprocessed data if available, otherwise preprocesses, balances the training set
    by oversampling the minority class, and saves the results.
    Includes robust path checking and detailed logging.
    """
    logging.info("--- Starting Data Loading/Preprocessing ---")
    if abs(train_split + val_split + test_split - 1.0) > 1e-6:
        raise ValueError(f"Train ({train_split}), validation ({val_split}), and test ({test_split}) splits must sum to 1.0")
    if not PATHS_VERIFIED: # Check verification status from earlier
         raise RuntimeError("Dataset path verification failed earlier. Cannot proceed.")

    # Define file paths for preprocessed data (more descriptive names)
    split_str = f"train{int(train_split*100)}_val{int(val_split*100)}_test{int(test_split*100)}"
    train_file = os.path.join(preprocessed_dir, f"train_balanced_{split_str}_seed{seed}.npz")
    val_file = os.path.join(preprocessed_dir, f"val_{split_str}_seed{seed}.npz")
    test_file = os.path.join(preprocessed_dir, f"test_{split_str}_seed{seed}.npz")
    metadata_file = os.path.join(preprocessed_dir, f"metadata_{split_str}_seed{seed}.npz")

    os.makedirs(preprocessed_dir, exist_ok=True)
    logging.info(f"Preprocessed data directory: {preprocessed_dir}")

    if os.path.exists(train_file) and os.path.exists(val_file) and os.path.exists(test_file) and os.path.exists(metadata_file):
        logging.info("Found existing preprocessed files. Loading data...")
        try:
            with np.load(train_file) as data:
                train_images, train_labels = data['images'], data['labels']
            with np.load(val_file) as data:
                val_images, val_labels = data['images'], data['labels']
            with np.load(test_file) as data:
                test_images, test_labels = data['images'], data['labels']
            with np.load(metadata_file) as data:
                loaded_num_classes = data['num_classes']
                loaded_img_size = data['img_size']
                loaded_seed = data['seed']
                loaded_fraction = data.get('dataset_fraction', 1.0) # Use get for backward compatibility

            # --- Verification of Loaded Data ---
            logging.info("Verifying loaded data parameters...")
            if int(loaded_num_classes) != config.NUM_CLASSES:
                 logging.warning(f"Loaded data has {loaded_num_classes} classes, but config expects {config.NUM_CLASSES}. Using loaded value.")
                 config.NUM_CLASSES = int(loaded_num_classes)
            if int(loaded_img_size) != img_size:
                 logging.warning(f"Loaded data has image size {loaded_img_size}, but config expects {img_size}. Using loaded value.")
                 config.IMG_SIZE = int(loaded_img_size) # Update config
            if int(loaded_seed) != seed:
                logging.warning(f"Loaded data was created with seed {loaded_seed}, but current seed is {seed}.")
            if float(loaded_fraction) != dataset_fraction:
                 logging.warning(f"Loaded data used fraction {loaded_fraction}, but current fraction is {dataset_fraction}.")

            logging.info("Loaded data shapes:")
            logging.info(f"  Train: Images={train_images.shape}, Labels={train_labels.shape}")
            logging.info(f"  Validation: Images={val_images.shape}, Labels={val_labels.shape}")
            logging.info(f"  Test: Images={test_images.shape}, Labels={test_labels.shape}")
            logging.info(f"Successfully loaded preprocessed data. Skipping preprocessing.")
            return train_images, train_labels, val_images, val_labels, test_images, test_labels

        except Exception as e:
            logging.error(f"Error loading preprocessed files: {e}. Attempting to re-process...", exc_info=True)
            # Clean up potentially corrupted files before reprocessing
            for f in [train_file, val_file, test_file, metadata_file]:
                 if os.path.exists(f): os.remove(f)

    # --- Preprocessing Steps ---
    logging.info("Preprocessing dataset from image files...")
    authentic_dir = os.path.join(data_path, "Au")
    tampered_dir = os.path.join(data_path, "Tp")

    # Glob patterns for common image types (case-insensitive)
    patterns = ['*.[jJ][pP][gG]', '*.[jJ][pP][eE][gG]', '*.[pP][nN][gG]', '*.[bB][mM][pP]', '*.[tT][iI][fF]', '*.[tT][iI][fF][fF]']
    authentic_paths = []
    for pattern in patterns:
        authentic_paths.extend(glob.glob(os.path.join(authentic_dir, pattern)))

    tampered_paths = []
    for pattern in patterns:
        tampered_paths.extend(glob.glob(os.path.join(tampered_dir, pattern)))

    num_auth_found = len(authentic_paths)
    num_tamp_found = len(tampered_paths)
    logging.info(f"Glob found {num_auth_found} potential authentic files.")
    logging.info(f"Glob found {num_tamp_found} potential tampered files.")

    if num_auth_found == 0 and num_tamp_found == 0:
        raise RuntimeError(f"Glob did not find any files matching patterns {patterns} in '{authentic_dir}' or '{tampered_dir}'. Check paths and file extensions.")

    if dataset_fraction < 1.0:
        logging.info(f"Sampling {dataset_fraction*100:.1f}% of the found files (before loading).")
        authentic_paths = random.sample(authentic_paths, int(num_auth_found * dataset_fraction))
        tampered_paths = random.sample(tampered_paths, int(num_tamp_found * dataset_fraction))
        logging.info(f"Reduced to {len(authentic_paths)} authentic and {len(tampered_paths)} tampered paths for processing.")

    all_image_paths = authentic_paths + tampered_paths
    all_labels = [0] * len(authentic_paths) + [1] * len(tampered_paths) # 0: Authentic, 1: Tampered

    images = []
    labels = []
    loaded_count = 0
    skipped_count = 0
    skipped_reasons: Dict[str, int] = {}

    # --- Image Loading Loop with Robustness ---
    logging.info(f"Attempting to load and resize {len(all_image_paths)} images to {img_size}x{img_size}...")
    for img_path, label in tqdm(zip(all_image_paths, all_labels), total=len(all_image_paths), desc="Loading & Resizing Images"):
        try:
            img = Image.open(img_path)
            # Ensure conversion to RGB *after* opening
            if img.mode != 'RGB':
                img = img.convert('RGB')

            img = img.resize((img_size, img_size), Image.Resampling.LANCZOS) # High-quality resize
            img_np = np.array(img, dtype=np.uint8) # Convert to numpy, ensuring uint8

            # --- Validity Checks ---
            if img_np.ndim != 3 or img_np.shape != (img_size, img_size, 3):
                reason = f"Unexpected shape {img_np.shape}"
                logging.warning(f"Skipping {os.path.basename(img_path)}: {reason}")
                skipped_count += 1
                skipped_reasons[reason] = skipped_reasons.get(reason, 0) + 1
                continue
            if img_np.size == 0:
                 reason = "Empty image data"
                 logging.warning(f"Skipping {os.path.basename(img_path)}: {reason}")
                 skipped_count += 1
                 skipped_reasons[reason] = skipped_reasons.get(reason, 0) + 1
                 continue

            images.append(img_np)
            labels.append(label)
            loaded_count += 1

        except (IOError, OSError, Image.DecompressionBombError) as e:
            reason = f"PIL/IO error: {type(e).__name__}"
            logging.warning(f"Skipping {os.path.basename(img_path)} due to {reason}: {e}")
            skipped_count += 1
            skipped_reasons[reason] = skipped_reasons.get(reason, 0) + 1
        except Exception as e:
            reason = f"Unexpected error: {type(e).__name__}"
            logging.error(f"Unexpected error loading {os.path.basename(img_path)}: {e}", exc_info=True) # Log traceback for unexpected
            skipped_count += 1
            skipped_reasons[reason] = skipped_reasons.get(reason, 0) + 1

    logging.info(f"Image loading finished. Successfully loaded: {loaded_count}, Skipped: {skipped_count}")
    if skipped_count > 0:
        logging.warning("Skipped image reasons:")
        for reason, count in skipped_reasons.items():
             logging.warning(f"  - {reason}: {count} times")

    if not images: # Check if the list is empty *after* the loop
        raise RuntimeError(f"No images were loaded successfully after processing {len(all_image_paths)} found paths. Check logs for reasons.")

    images = np.array(images, dtype=np.uint8) # Ensure uint8 array
    labels = np.array(labels, dtype=np.int64) # Use int64 for labels

    actual_num_classes = len(np.unique(labels))
    if actual_num_classes != config.NUM_CLASSES:
         logging.warning(f"Found {actual_num_classes} unique labels in loaded data, but config expected {config.NUM_CLASSES}. Updating config.")
         config.NUM_CLASSES = actual_num_classes
    if actual_num_classes < 2:
         raise RuntimeError(f"Only {actual_num_classes} class found in the loaded data. Need at least 2 for classification.")

    logging.info(f"Total images loaded into numpy array: {images.shape}. Labels: {labels.shape}")

    # --- Split into Train/Val/Test ---
    logging.info(f"Splitting data using ratios: Train={train_split}, Val={val_split}, Test={test_split} (Stratified)")
    # Split test set first
    remaining_images, test_images, remaining_labels, test_labels = train_test_split(
        images, labels,
        test_size=test_split,
        stratify=labels,
        random_state=seed
    )
    # Split train and validation from the remainder
    val_split_adjusted = val_split / (train_split + val_split) # Adjust val split for the remaining data
    train_images_unbalanced, val_images, train_labels_unbalanced, val_labels = train_test_split(
        remaining_images, remaining_labels,
        test_size=val_split_adjusted,
        stratify=remaining_labels,
        random_state=seed
    )

    logging.info("Dataset split (before balancing training set):")
    logging.info(f"  Train (unbalanced): {len(train_images_unbalanced)} images. Distribution: {np.bincount(train_labels_unbalanced)}")
    logging.info(f"  Validation: {len(val_images)} images. Distribution: {np.bincount(val_labels)}")
    logging.info(f"  Test: {len(test_images)} images. Distribution: {np.bincount(test_labels)}")

    # --- Balance the Training Set via Oversampling ---
    logging.info("Balancing the training set using Oversampling...")
    unique_labels_train, counts_train = np.unique(train_labels_unbalanced, return_counts=True)

    if len(unique_labels_train) < 2:
        logging.warning("Training set contains only one class after splitting. Skipping balancing.")
        train_images = train_images_unbalanced
        train_labels = train_labels_unbalanced
    elif counts_train[0] == counts_train[1]:
        logging.info("Training set is already balanced.")
        train_images = train_images_unbalanced
        train_labels = train_labels_unbalanced
    else:
        minority_label = unique_labels_train[np.argmin(counts_train)]
        majority_label = unique_labels_train[np.argmax(counts_train)]
        n_minority = counts_train.min()
        n_majority = counts_train.max()
        n_to_add = n_majority - n_minority

        logging.info(f"  Majority class ({majority_label}): {n_majority} samples.")
        logging.info(f"  Minority class ({minority_label}): {n_minority} samples.")
        logging.info(f"  Oversampling minority class by adding {n_to_add} samples.")

        minority_indices = np.where(train_labels_unbalanced == minority_label)[0]
        # Use numpy's random generator seeded earlier for reproducibility
        np_rng = np.random.RandomState(seed)
        oversample_indices = np_rng.choice(minority_indices, size=n_to_add, replace=True)

        oversampled_images = train_images_unbalanced[oversample_indices]
        oversampled_labels = train_labels_unbalanced[oversample_indices]

        # Concatenate and shuffle
        train_images = np.concatenate((train_images_unbalanced, oversampled_images), axis=0)
        train_labels = np.concatenate((train_labels_unbalanced, oversampled_labels), axis=0)
        train_images, train_labels = shuffle(train_images, train_labels, random_state=seed)

    logging.info("Training set balanced:")
    logging.info(f"  Total training images: {len(train_images)}")
    logging.info(f"  Balanced train distribution: {np.bincount(train_labels)}")

    # --- Save Processed Data ---
    logging.info(f"Saving preprocessed data to {preprocessed_dir}...")
    try:
        np.savez_compressed(train_file, images=train_images, labels=train_labels)
        np.savez_compressed(val_file, images=val_images, labels=val_labels)
        np.savez_compressed(test_file, images=test_images, labels=test_labels)
        np.savez_compressed(metadata_file,
                            num_classes=config.NUM_CLASSES,
                            img_size=img_size,
                            seed=seed,
                            dataset_fraction=dataset_fraction,
                            train_split=train_split,
                            val_split=val_split,
                            test_split=test_split)
        logging.info("Successfully saved preprocessed data files.")
    except Exception as e:
        logging.error(f"Failed to save preprocessed data: {e}", exc_info=True)
        # Don't raise here, allow the function to return the in-memory data

    logging.info("--- Finished Data Loading/Preprocessing ---")
    return train_images, train_labels, val_images, val_labels, test_images, test_labels


# --- Data Transformations ---
train_transforms = transforms.Compose([
    # Resize slightly larger than target, then RandomCrop
    # transforms.Resize(config.IMG_SIZE + 32), # Example: Resize to 256 for 224 crop
    # transforms.RandomCrop(config.IMG_SIZE),
    # Or use RandomResizedCrop carefully
    transforms.RandomResizedCrop(config.IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1)), # Tweak scale/ratio
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1), # Moderate jitter
    transforms.RandomHorizontalFlip(p=0.5),
    # transforms.RandomVerticalFlip(p=0.2), # Optional, maybe less useful
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)), # Slight translation
    transforms.ToTensor(),
    transforms.Normalize(mean=config.MEAN, std=config.STD)
])

val_test_transforms = transforms.Compose([
    transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=config.MEAN, std=config.STD)
])
logging.info("Data transformations defined.")

# === MODEL COMPONENTS (CBAM, PositionalEncoding, TransformerLayer) ===

# CBAM Module
class ChannelAttention(nn.Module):
    # ... (keep implementation as before) ...
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc1 = nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)

        self.sigmoid = nn.Sigmoid()
        self._initialize_weights()

    def _initialize_weights(self):
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.kaiming_normal_(self.fc2.weight, mode='fan_in', nonlinearity='sigmoid') # Kaiming for layer before sigmoid

    def forward(self, x):
        avg_out = self.fc2(self.relu1(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu1(self.fc1(self.max_pool(x))))
        out = avg_out + max_out
        return self.sigmoid(out)

class SpatialAttention(nn.Module):
    # ... (keep implementation as before) ...
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        assert kernel_size in (3, 7), 'kernel size must be 3 or 7'
        padding = 3 if kernel_size == 7 else 1

        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()
        self._initialize_weights()

    def _initialize_weights(self):
         nn.init.xavier_uniform_(self.conv1.weight)

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        x_out = self.conv1(x_cat)
        return self.sigmoid(x_out)

class CBAM(nn.Module):
    # ... (keep implementation as before) ...
    def __init__(self, channels, reduction_ratio=16, kernel_size=7, dropout_rate=0.0): # Default dropout 0 for CBAM itself
        super(CBAM, self).__init__()
        self.channelattention = ChannelAttention(channels, ratio=reduction_ratio)
        self.spatialattention = SpatialAttention(kernel_size=kernel_size)
        self.dropout = nn.Dropout(dropout_rate) if dropout_rate > 0 else nn.Identity()

    def forward(self, x):
        x_out = x * self.channelattention(x)
        x_out = x_out * self.spatialattention(x_out)
        return self.dropout(x_out)


# Positional Encoding
class PositionalEncoding(nn.Module):
    # ... (keep implementation as before, ensure math is imported) ...
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 10): # max_len only needs to be 2 (Swin, EffNet)
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        # Corrected div_term calculation
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        # Initialize pe tensor: shape (1, max_len, d_model) for batch dimension broadcasting
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        # Handle odd d_model case for cosine part
        pe[0, :, 1::2] = torch.cos(position * div_term[:pe[0, :, 1::2].shape[1]]) # Slice div_term if needed

        self.register_buffer('pe', pe) # Register as buffer, not parameter

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Tensor, shape [batch_size, seq_len, embedding_dim]
        """
        # Add positional encoding up to the sequence length of x
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# Custom Transformer Encoder Layer to output attention weights
class TransformerEncoderLayerWithAttention(TransformerEncoderLayer):
    # ... (keep implementation as before, ensure imports OK) ...
    # NOTE: Using norm_first=True is often more stable in Transformers
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1, activation=F.gelu, # Use GELU
                 layer_norm_eps=1e-5, batch_first=True, norm_first=True, # Set norm_first=True
                 device=None, dtype=None) -> None:
        factory_kwargs = {'device': device, 'dtype': dtype}
        super().__init__(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout,
                         activation=activation, layer_norm_eps=layer_norm_eps, batch_first=batch_first,
                         norm_first=norm_first, **factory_kwargs)
        # Re-initialize MHA to ensure batch_first is respected if parent class doesn't handle it fully
        # (Though modern PyTorch versions should handle this correctly in super().__init__)
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=batch_first,
                                                **factory_kwargs)

    # Override forward to ensure attention weights are returned
    # This implementation mirrors the source code for TransformerEncoderLayer
    # adapting it to explicitly return attention weights.
    def forward(self, src: torch.Tensor, src_mask: Optional[torch.Tensor] = None,
                src_key_padding_mask: Optional[torch.Tensor] = None, is_causal: bool = False) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        # Check for causal mask incompatibility (added in PyTorch 1.12+)
        # You might need to adjust this check based on your PyTorch version
        # if src_mask is not None and hasattr(self.self_attn, "need_weights") and self.self_attn.need_weights == False:
        #     logging.warning("src_mask provided but self_attn.need_weights=False. Attention weights might be incorrect.")
        # if is_causal and src_mask is not None:
        #      raise RuntimeError("Cannot provide both src_mask and is_causal=True")
        # causal_mask = None
        # if is_causal:
        #      causal_mask = self._generate_causal_mask(src.size(-2), src.size(-3), device=src.device, dtype=src.dtype) # Check function exists or implement


        attn_weights: Optional[torch.Tensor] = None # Initialize attn_weights

        # --- Logic based on norm_first ---
        if self.norm_first:
             # 1. Norm + Multihead Attention + Dropout + Residual
             norm_src = self.norm1(src)
             # Request attention weights, handle potential causal mask
             sa_output, attn_weights = self._sa_block(norm_src, src_mask, src_key_padding_mask) # Pass potential causal mask if using newer PyTorch
             x = src + sa_output # Residual connection 1

             # 2. Norm + Feedforward + Dropout + Residual
             norm_x = self.norm2(x)
             ff_output = self._ff_block(norm_x)
             output = x + ff_output # Residual connection 2
        else: # Norm last (less common now)
             # 1. Multihead Attention + Dropout + Residual + Norm
             sa_output, attn_weights = self._sa_block(src, src_mask, src_key_padding_mask) # Pass potential causal mask
             x = self.norm1(src + sa_output) # Residual connection 1 + Norm 1

             # 2. Feedforward + Dropout + Residual + Norm
             ff_output = self._ff_block(x)
             output = self.norm2(x + ff_output) # Residual connection 2 + Norm 2

        return output, attn_weights # Return output and attention weights

    # Helper function for self-attention block (encapsulates MHA call)
    def _sa_block(self, x: torch.Tensor, attn_mask: Optional[torch.Tensor], key_padding_mask: Optional[torch.Tensor]) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        # Ensure need_weights=True to get attention weights
        # Handle potential causal mask argument depending on PyTorch version
        try: # Newer PyTorch MHA might take is_causal in forward
             mha_output, attn_weights = self.self_attn(x, x, x,
                                                attn_mask=attn_mask,
                                                key_padding_mask=key_padding_mask,
                                                need_weights=True,
                                                average_attn_weights=False) # Return weights per head
        except TypeError: # Older PyTorch MHA might not have average_attn_weights
             mha_output, attn_weights = self.self_attn(x, x, x,
                                                attn_mask=attn_mask,
                                                key_padding_mask=key_padding_mask,
                                                need_weights=True)

        return self.dropout1(mha_output), attn_weights

    # Helper function for feedforward block (logic from nn.TransformerEncoderLayer)
    def _ff_block(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear2(self.dropout(self.activation(self.linear1(x))))
        return self.dropout2(x)


# === Main Late Fusion Model ===
# === Main Late Fusion Model ===
class LateFusionWithTransformer(nn.Module):
    def __init__(self, num_classes, embed_dim=config.EMBED_DIM, num_heads=config.NUM_HEADS, num_layers=config.NUM_LAYERS, dropout=config.DROPOUT):
        super(LateFusionWithTransformer, self).__init__()
        self.device = DEVICE # Store device

        logging.info("Initializing LateFusionWithTransformer model...")

        # --- Get Feature Dimensions Correctly ---
        try:
            logging.info("Probing Swin Transformer (swin_tiny_patch4_window7_224) for feature dimension...")
            _swin_temp = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True)
            swin_feat_dim = _swin_temp.head.in_features
            logging.info(f"  - Swin feature dimension detected: {swin_feat_dim}")
            del _swin_temp # Free memory
        except Exception as e:
            logging.error(f"Could not probe Swin feature dimension: {e}", exc_info=True)
            raise # Re-raise the error to stop initialization

        try:
            logging.info("Probing EfficientNet (efficientnet_b0) for feature dimension...")
            _effnet_temp = timm.create_model('efficientnet_b0', pretrained=True)
            eff_feat_dim = _effnet_temp.classifier.in_features
            logging.info(f"  - EfficientNet feature dimension detected: {eff_feat_dim}")
            del _effnet_temp # Free memory
        except Exception as e:
            logging.error(f"Could not probe EfficientNet feature dimension: {e}", exc_info=True)
            raise # Re-raise the error

        # --- Load Models for Feature Extraction (num_classes=0 replaces head) ---
        logging.info("Loading Swin Transformer for feature extraction...")
        self.swin = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=0)
        logging.info("Loading EfficientNet for feature extraction...")
        self.efficientnet = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)

        # --- CBAM for EfficientNet features ---
        self.cbam = CBAM(channels=eff_feat_dim, dropout_rate=0.0)
        logging.info(f"  - CBAM module initialized for {eff_feat_dim} channels.")

        # --- Projection Layers (Project features to embed_dim) ---
        self.swin_proj = nn.Linear(swin_feat_dim, embed_dim)
        self.eff_proj = nn.Linear(eff_feat_dim, embed_dim)
        logging.info(f"  - Projection layers initialized: Swin({swin_feat_dim})->{embed_dim}, EffNet({eff_feat_dim})->{embed_dim}")

        # --- Shared Dropout for projections ---
        self.proj_dropout = nn.Dropout(dropout)

        # --- Positional Encoding ---
        self.positional_encoding = PositionalEncoding(embed_dim, dropout=dropout, max_len=2)

        # --- Transformer Encoder using the custom layer ---
        logging.info(f"Initializing Transformer Encoder: {num_layers} layers, {num_heads} heads, embed_dim={embed_dim}, dropout={dropout}")
        encoder_layer = TransformerEncoderLayerWithAttention(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation=F.gelu
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # --- Final Classifier ---
        self.classifier_dropout = nn.Dropout(dropout + 0.1)
        self.classifier = nn.Linear(embed_dim, num_classes)
        logging.info(f"  - Final classifier initialized: {embed_dim} -> {num_classes}")

        self.to(self.device)
        self._initialize_weights()
        logging.info("Model initialization complete.")

    def _initialize_weights(self):
        # ... (rest of the __init__ and the _initialize_weights method remains the same) ...
        logging.info("Initializing weights for projection and classifier layers (Xavier Uniform)...")
        for layer in [self.swin_proj, self.eff_proj, self.classifier]:
            nn.init.xavier_uniform_(layer.weight)
            if layer.bias is not None:
                 nn.init.constant_(layer.bias, 0)

    # ... (forward method remains the same) ...
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, List[Optional[torch.Tensor]]]:
        # x: (batch, channels, height, width)

        # --- Swin Transformer Branch ---
        # Use autocast here if using mixed precision for backbone inference too
        with autocast(enabled=torch.cuda.is_available()):
             swin_features = self.swin(x) # (batch, swin_feat_dim)
        swin_proj = self.swin_proj(swin_features) # (batch, embed_dim)

        # --- EfficientNet Branch ---
        with autocast(enabled=torch.cuda.is_available()):
             eff_features_map = self.efficientnet.forward_features(x) # (batch, eff_feat_dim, H, W)
             eff_features_map_att = self.cbam(eff_features_map) # Apply CBAM
             # Global Average Pooling
             eff_features = F.adaptive_avg_pool2d(eff_features_map_att, (1, 1)).flatten(start_dim=1) # (batch, eff_feat_dim)
        eff_proj = self.eff_proj(eff_features) # (batch, embed_dim)

        # --- Fusion Preparation ---
        # Stack features for Transformer: (batch, seq_len=2, embed_dim)
        combined = torch.stack([swin_proj, eff_proj], dim=1)
        combined = self.proj_dropout(combined) # Apply dropout *after* projection

        # Add positional encoding
        combined = self.positional_encoding(combined) # (batch, seq_len=2, embed_dim)

        # --- Transformer Encoder ---
        transformer_output = combined
        all_attn_weights: List[Optional[torch.Tensor]] = [] # Store attention weights per layer

        # Iterate through layers explicitly to capture weights
        for layer in self.transformer_encoder.layers:
            # Pass through the layer; custom layer returns (output, attn_weights)
            if isinstance(layer, TransformerEncoderLayerWithAttention):
                transformer_output, attn_weights = layer(transformer_output)
                all_attn_weights.append(attn_weights)
            else: # Fallback if standard layers were somehow used
                transformer_output = layer(transformer_output)
                all_attn_weights.append(None) # Indicate no attention weights available

        # --- Classification Head ---
        # Aggregate features from Transformer output (e.g., mean pooling over sequence dimension)
        # Input shape: (batch, seq_len=2, embed_dim)
        fused_features = torch.mean(transformer_output, dim=1) # (batch, embed_dim)

        # Apply dropout and classify
        fused_features = self.classifier_dropout(fused_features)
        output_logits = self.classifier(fused_features) # (batch, num_classes)

        # Return logits and the list of attention weights from each layer
        return output_logits, all_attn_weights

    # Optional: Method to get backbone features (no gradients)
    @torch.no_grad()
    def get_backbone_features(self, x):
         self.eval() # Set to eval mode
         swin_features = self.swin(x)
         eff_features_map = self.efficientnet.forward_features(x)
         eff_features_map_att = self.cbam(eff_features_map)
         eff_features = F.adaptive_avg_pool2d(eff_features_map_att, (1, 1)).flatten(1)
         return swin_features, eff_features

# === Plotting Functions ===

def plot_attention_weights(all_attn_weights_list: List[List[Optional[torch.Tensor]]], model_name: str, num_samples_plotted: int):
    """Plots the average attention weights of the Transformer layers."""
    # ... (keep implementation as before, ensure robustness to None weights) ...
    if not all_attn_weights_list:
        logging.warning("No attention weights provided to plot_attention_weights.")
        return

    # Determine number of layers and check consistency
    num_layers = 0
    if all_attn_weights_list[0] is not None:
        num_layers = len(all_attn_weights_list[0])

    if num_layers == 0:
        logging.warning("No Transformer layers with attention weights found in the first batch.")
        return

    # Aggregate attention weights across batches for each layer
    avg_attn_weights_per_layer = []
    for layer_idx in range(num_layers):
        layer_weights_list = []
        # Collect weights for this layer from all batches
        for batch_weights in all_attn_weights_list:
            if batch_weights and len(batch_weights) > layer_idx and batch_weights[layer_idx] is not None:
                # Detach, move to CPU, convert to numpy
                layer_weights_list.append(batch_weights[layer_idx].detach().cpu().numpy())

        if not layer_weights_list:
            logging.warning(f"No valid attention weights found for Layer {layer_idx + 1}.")
            avg_attn_weights_per_layer.append(None)
            continue

        # Concatenate weights from all batches for this layer
        try:
            concatenated_weights = np.concatenate(layer_weights_list, axis=0)
             # Expected shape: (total_samples, num_heads, seq_len, seq_len)
        except ValueError as e:
             logging.error(f"Error concatenating weights for layer {layer_idx+1}: {e}. Shapes might be inconsistent.")
             # Attempt to log shapes for debugging
             shapes = [w.shape for w in layer_weights_list]
             logging.error(f"Weight shapes: {shapes}")
             avg_attn_weights_per_layer.append(None)
             continue


        # Average over samples (axis 0) and heads (axis 1)
        if concatenated_weights.ndim == 4:
             avg_weights = np.mean(concatenated_weights, axis=(0, 1))
        elif concatenated_weights.ndim == 3: # Might happen if num_heads=1 or averaged earlier
             avg_weights = np.mean(concatenated_weights, axis=0)
        else:
            logging.warning(f"Unexpected attention weight dimensions ({concatenated_weights.ndim}) for Layer {layer_idx + 1}. Cannot average.")
            avg_attn_weights_per_layer.append(None)
            continue

        avg_attn_weights_per_layer.append(avg_weights) # Shape (seq_len, seq_len)


    # --- Plotting ---
    feature_names = ['Swin', 'EfficientNet'] # Order matters, must match torch.stack order
    seq_len = len(feature_names)

    for layer_idx, avg_weights in enumerate(avg_attn_weights_per_layer):
        if avg_weights is None:
            logging.warning(f"Skipping plot for Layer {layer_idx + 1} due to missing weights.")
            continue
        if avg_weights.shape != (seq_len, seq_len):
            logging.warning(f"Skipping plot for Layer {layer_idx + 1}: expected shape {(seq_len, seq_len)}, got {avg_weights.shape}")
            continue

        plt.figure(figsize=(8, 6))
        sns.heatmap(avg_weights, annot=True, cmap="viridis", fmt=".3f",
                    xticklabels=feature_names, yticklabels=feature_names,
                    linewidths=.5) # Add lines between cells
        plt.xlabel("Key/Value Source")
        plt.ylabel("Query Source")
        plt.title(f"Layer {layer_idx + 1} Avg Attention Weights ({num_samples_plotted} samples)\nModel: {model_name}")
        plt.tight_layout()
        save_path = f'attention_layer_{layer_idx + 1}_{model_name}.png'
        try:
            plt.savefig(save_path)
            logging.info(f"Saved attention plot: {save_path}")
        except Exception as e:
            logging.error(f"Failed to save attention plot '{save_path}': {e}")
        plt.show()

def plot_training_history(history: Dict[str, List[float]], phase: int, model_name: str):
    """Plots training and validation loss and accuracy curves."""
    # ... (keep implementation as before) ...
    epochs = range(1, len(history.get('train_loss', [])) + 1)
    if not epochs:
        logging.warning(f"No history data found for phase {phase} to plot.")
        return

    plt.figure(figsize=(14, 6))

    # Loss Curves
    plt.subplot(1, 2, 1)
    if 'train_loss' in history: plt.plot(epochs, history['train_loss'], 'bo-', label='Train Loss')
    if 'val_loss' in history: plt.plot(epochs, history['val_loss'], 'ro-', label='Validation Loss')
    plt.title(f'Phase {phase} Loss - {model_name}')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)

    # Accuracy Curves
    plt.subplot(1, 2, 2)
    if 'train_acc' in history: plt.plot(epochs, history['train_acc'], 'bo-', label='Train Accuracy')
    if 'val_acc' in history: plt.plot(epochs, history['val_acc'], 'ro-', label='Validation Accuracy')
    plt.title(f'Phase {phase} Accuracy - {model_name}')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    save_path = f'training_curves_phase_{phase}_{model_name}.png'
    try:
        plt.savefig(save_path)
        logging.info(f"Saved training curves plot: {save_path}")
    except Exception as e:
        logging.error(f"Failed to save training curves plot '{save_path}': {e}")
    plt.show()


# === Training & Validation Functions ===

# Use GradScaler for mixed precision training
scaler = GradScaler(enabled=(DEVICE.type == 'cuda'))
logging.info(f"Gradient Scaler enabled: {scaler.is_enabled()}")

def train_one_epoch(model: nn.Module, dataloader: DataLoader, criterion: nn.Module, optimizer: optim.Optimizer, device: torch.device, scheduler: Optional[optim.lr_scheduler._LRScheduler], clip_value: float) -> Tuple[float, float]:
    """Trains the model for one epoch."""
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    start_time = time.time()

    progress_bar = tqdm(dataloader, desc="Training", leave=False, dynamic_ncols=True)
    for batch_idx, (inputs, labels) in enumerate(progress_bar):
        # Handle potential dummy data from dataset __getitem__ error
        if -1 in labels:
             logging.warning(f"Skipping batch {batch_idx} due to dummy data (label -1 found).")
             continue

        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True) # More efficient zeroing

        # Mixed precision context
        with autocast(enabled=scaler.is_enabled()):
            outputs, _ = model(inputs) # Ignore attention weights during training
            loss = criterion(outputs, labels)

        # Scale loss, backward pass, and gradient clipping
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer) # Unscale gradients before clipping
        nn.utils.clip_grad_norm_(model.parameters(), clip_value)
        scaler.step(optimizer) # Optimizer step (updates parameters)
        scaler.update() # Update scaler for next iteration

        # Step the scheduler (if one is provided) - step per batch for CosineAnnealingLR with T_max=total_steps
        if scheduler:
             scheduler.step()

        # --- Accumulate Metrics ---
        batch_loss = loss.item()
        running_loss += batch_loss * inputs.size(0)

        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

        # Update progress bar postfix
        progress_bar.set_postfix(
            loss=f"{batch_loss:.4f}",
            acc=f"{correct_predictions/total_samples:.4f}",
            lr=f"{optimizer.param_groups[0]['lr']:.1e}" # Show current LR
        )

    epoch_duration = time.time() - start_time
    # Ensure total_samples > 0 to avoid division by zero if all batches were skipped
    if total_samples == 0:
        logging.warning("Epoch finished but no samples were processed successfully.")
        return 0.0, 0.0

    epoch_loss = running_loss / total_samples
    epoch_acc = correct_predictions / total_samples
    logging.info(f"Train Epoch Summary: Duration={epoch_duration:.2f}s, Avg Loss={epoch_loss:.4f}, Avg Acc={epoch_acc:.4f}")
    return epoch_loss, epoch_acc


@torch.no_grad() # Decorator for validation (no gradients needed)
def validate(model: nn.Module, dataloader: DataLoader, criterion: nn.Module, device: torch.device) -> Tuple[float, float]:
    """Validates the model on the validation set."""
    model.eval() # Set model to evaluation mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    all_preds_val = []
    all_labels_val = []
    start_time = time.time()

    progress_bar = tqdm(dataloader, desc="Validation", leave=False, dynamic_ncols=True)
    for inputs, labels in progress_bar:
        if -1 in labels: # Skip dummy data
             continue
        inputs, labels = inputs.to(device), labels.to(device)

        # Use autocast for potentially faster inference (though less critical than training)
        with autocast(enabled=scaler.is_enabled()):
            outputs, _ = model(inputs) # Ignore attention weights
            loss = criterion(outputs, labels)

        batch_loss = loss.item()
        running_loss += batch_loss * inputs.size(0)

        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

        # Store predictions and labels for detailed report later
        all_preds_val.extend(predicted.cpu().numpy())
        all_labels_val.extend(labels.cpu().numpy())

        # Update progress bar postfix
        progress_bar.set_postfix(loss=f"{batch_loss:.4f}", acc=f"{correct_predictions/total_samples:.4f}")

    epoch_duration = time.time() - start_time
    if total_samples == 0:
        logging.warning("Validation finished but no samples were processed successfully.")
        return float('inf'), 0.0 # Return high loss, zero accuracy

    epoch_loss = running_loss / total_samples
    epoch_acc = correct_predictions / total_samples

    # Print validation classification report
    try:
        logging.info("\n--- Validation Set Report ---")
        report = classification_report(all_labels_val, all_preds_val, target_names=['Authentic', 'Tampered'], digits=4, zero_division=0)
        logging.info("\n" + report)
        print("\nValidation Classification Report:\n", report) # Also print to console
        logging.info("--- End Validation Set Report ---")
    except Exception as e:
        logging.error(f"Could not generate validation classification report: {e}")


    logging.info(f"Validation Epoch Summary: Duration={epoch_duration:.2f}s, Avg Loss={epoch_loss:.4f}, Avg Acc={epoch_acc:.4f}")
    return epoch_loss, epoch_acc


def train_model(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, criterion: nn.Module, optimizer: optim.Optimizer, scheduler: Optional[optim.lr_scheduler._LRScheduler], num_epochs: int, device: torch.device, patience: int, clip_value: float, phase: int, model_save_name: str) -> Tuple[nn.Module, Dict[str, List[float]]]:
    """Trains the model for a given phase with early stopping."""
    best_val_loss = float('inf')
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    logging.info(f"--- Starting Training Phase {phase} for {num_epochs} Epochs ---")
    logging.info(f"Model state will be saved to: {model_save_name}")
    logging.info(f"Early stopping patience: {patience}")

    total_start_time = time.time()

    for epoch in range(num_epochs):
        epoch_num = epoch + 1
        epoch_start_time = time.time()
        logging.info(f"===> Epoch {epoch_num}/{num_epochs}, Phase {phase} <===")

        # --- Training ---
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, scheduler, clip_value)

        # --- Validation ---
        val_loss, val_acc = validate(model, val_loader, criterion, device)

        # --- Logging & History ---
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        epoch_duration = time.time() - epoch_start_time
        current_lr = optimizer.param_groups[0]['lr']

        log_msg = (f"Epoch {epoch_num} Summary: "
                   f"Time={epoch_duration:.2f}s, "
                   f"LR={current_lr:.2e}, "
                   f"Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, "
                   f"Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")
        logging.info(log_msg)
        print(log_msg) # Print summary to console as well

        # --- Checkpoint Saving & Early Stopping (based on validation loss) ---
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            try:
                torch.save(model.state_dict(), model_save_name)
                logging.info(f"Validation loss improved to {best_val_loss:.4f}. Model saved to '{os.path.basename(model_save_name)}'")
            except Exception as e:
                logging.error(f"Error saving model: {e}")
        else:
            patience_counter += 1
            logging.info(f"Validation loss ({val_loss:.4f}) did not improve from best ({best_val_loss:.4f}). Patience: {patience_counter}/{patience}")
            if patience_counter >= patience:
                logging.warning("Early stopping triggered.")
                print(f"Early stopping triggered after {epoch_num} epochs.")
                break

        # Optional: Clear CUDA cache periodically if memory issues occur
        if device.type == 'cuda' and epoch_num % 5 == 0:
             torch.cuda.empty_cache()
             # logging.debug("Cleared CUDA cache.")

    total_training_time = time.time() - total_start_time
    logging.info(f"--- Finished Training Phase {phase}. Total Time: {total_training_time:.2f}s ---")

    # --- Load Best Model ---
    # Load the best model state found during this phase based on validation loss
    if os.path.exists(model_save_name):
        try:
            logging.info(f"Loading best model state from phase {phase}: '{os.path.basename(model_save_name)}'")
            model.load_state_dict(torch.load(model_save_name, map_location=device))
        except Exception as e:
            logging.error(f"Error loading best model state '{model_save_name}': {e}. Using model state from last epoch.")
    else:
        logging.warning(f"Best model file '{model_save_name}' not found after training phase {phase}. Using model state from the last epoch.")

    return model, history


# === Evaluation Function ===
@torch.no_grad()
def evaluate_model(model: nn.Module, test_loader: DataLoader, device: torch.device, num_classes: int, model_name: str) -> Tuple[List[List[Optional[torch.Tensor]]], int]:
    """Evaluates the final model on the test set and collects attention weights."""
    logging.info("\n--- Starting Final Evaluation on Test Set ---")
    model.eval()

    all_preds = []
    all_labels = []
    all_attention_weights_batches = [] # Store weights for each batch
    total_samples = 0
    start_time = time.time()

    progress_bar = tqdm(test_loader, desc="Testing", leave=False, dynamic_ncols=True)
    for inputs, labels in progress_bar:
        if -1 in labels: # Skip dummy data
             continue
        inputs, labels = inputs.to(device), labels.to(device)
        batch_size = inputs.size(0)

        # Get model output and attention weights
        with autocast(enabled=scaler.is_enabled()):
             outputs, attn_weights_batch = model(inputs)

        # Store weights for this batch (attn_weights_batch is a list of weights per layer)
        all_attention_weights_batches.append(attn_weights_batch)

        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        total_samples += batch_size

    eval_duration = time.time() - start_time
    logging.info(f"Test set evaluation finished in {eval_duration:.2f}s.")

    if total_samples == 0:
         logging.error("No samples were processed during test set evaluation.")
         return [], 0

    # --- Metrics Calculation & Display ---
    logging.info("\n--- Test Set Results ---")
    # Classification Report
    target_names = [f'Class_{i}' for i in range(num_classes)]
    if num_classes == 2: # Use specific names for binary case
        target_names = ['Authentic (0)', 'Tampered (1)']

    try:
        report = classification_report(all_labels, all_preds, target_names=target_names, digits=4, zero_division=0)
        logging.info("Classification Report:\n" + report)
        print("\nTest Set Classification Report:\n", report)
    except Exception as e:
        logging.error(f"Could not generate classification report: {e}")

    # --- Confusion Matrix (Counts) ---
    try:
        cm_counts = confusion_matrix(all_labels, all_preds)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm_counts, annot=True, fmt="d", cmap="Blues", xticklabels=target_names, yticklabels=target_names, annot_kws={"size": 12})
        plt.xlabel("Predicted Labels")
        plt.ylabel("True Labels")
        plt.title(f"Test Set Confusion Matrix (Counts)\nModel: {model_name}")
        plt.tight_layout()
        save_path_counts = f'test_confusion_matrix_counts_{model_name}.png'
        plt.savefig(save_path_counts)
        logging.info(f"Saved confusion matrix (counts): {save_path_counts}")
        plt.show()
    except Exception as e:
        logging.error(f"Could not generate or save confusion matrix (counts): {e}")

    # --- Confusion Matrix (Percentage) ---
    try:
        cm_percent = confusion_matrix(all_labels, all_preds, normalize='true')
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm_percent, annot=True, fmt=".2%", cmap="Greens", xticklabels=target_names, yticklabels=target_names, annot_kws={"size": 12})
        plt.xlabel("Predicted Labels")
        plt.ylabel("True Labels")
        plt.title(f"Test Set Confusion Matrix (Percentage)\nModel: {model_name}")
        plt.tight_layout()
        save_path_percent = f'test_confusion_matrix_percent_{model_name}.png'
        plt.savefig(save_path_percent)
        logging.info(f"Saved confusion matrix (percentage): {save_path_percent}")
        plt.show()
    except Exception as e:
        logging.error(f"Could not generate or save confusion matrix (percentage): {e}")

    logging.info("--- Finished Final Evaluation ---")
    # Return collected attention weights and the number of samples evaluated
    return all_attention_weights_batches, total_samples

# === Main Execution Block ===
if __name__ == "__main__":

    logging.info("========== Starting Experiment Run ==========")
    logging.info(f"Model Name: {config.MODEL_NAME}")
    logging.info(f"Configuration: {vars(config)}")

    # Ensure dataset paths were verified successfully before proceeding
    if not PATHS_VERIFIED:
        logging.critical("Dataset paths could not be verified. Exiting.")
        # Keep exit() here, as bad paths are unrecoverable
        exit()

    # --- Load or Preprocess Data ---
    # Assign default values in case of failure below
    train_images, train_labels, val_images, val_labels, test_images, test_labels = None, None, None, None, None, None
    train_loader, val_loader, test_loader = None, None, None # Initialize loaders to None
    data_loaded_successfully = False
    try:
        train_images, train_labels, val_images, val_labels, test_images, test_labels = load_or_preprocess_data(
            data_path=config.DATA_PATH,
            preprocessed_dir=config.PREPROCESSED_DIR,
            img_size=config.IMG_SIZE,
            dataset_fraction=config.DATASET_FRACTION,
            train_split=config.TRAIN_SIZE,
            val_split=config.VAL_SIZE,
            test_split=config.TEST_SIZE,
            seed=config.SEED
        )
        # Verify data shapes after loading/processing
        logging.info("Final data shapes:")
        logging.info(f"  Train: {train_images.shape}, {train_labels.shape}, dtype={train_images.dtype}, {train_labels.dtype}")
        logging.info(f"  Validation: {val_images.shape}, {val_labels.shape}, dtype={val_images.dtype}, {val_labels.dtype}")
        logging.info(f"  Test: {test_images.shape}, {test_labels.shape}, dtype={test_images.dtype}, {test_labels.dtype}")
        data_loaded_successfully = True # Mark success

    except (RuntimeError, ValueError, FileNotFoundError, TypeError) as e:
        logging.critical(f"Fatal error during data loading/preprocessing: {e}", exc_info=True)
        # Keep exit() here, as data loading failure is critical
        exit()

    # --- Create Datasets and DataLoaders ---
    datasets_created_successfully = False
    if data_loaded_successfully: # Only proceed if data is loaded
        try:
            logging.info("Creating Datasets...")
            train_dataset = CASIADataset(train_images, train_labels, train_transforms)
            val_dataset = CASIADataset(val_images, val_labels, val_test_transforms)
            test_dataset = CASIADataset(test_images, test_labels, val_test_transforms)
            logging.info("Datasets created successfully.")

            # Determine optimal number of workers
            num_workers = 0
            if DEVICE.type == 'cuda':
                 num_workers = min(os.cpu_count() // 2 if os.cpu_count() else 0, 4)
            logging.info(f"Using {num_workers} workers for DataLoaders.")

            logging.info("Creating DataLoaders...")
            train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True,
                                      pin_memory=True, num_workers=num_workers, drop_last=True)
            val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
                                    pin_memory=True, num_workers=num_workers)
            test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False,
                                     pin_memory=True, num_workers=num_workers)

            logging.info(f"DataLoaders created:")
            logging.info(f"  Train batches = {len(train_loader)} (batch size {config.BATCH_SIZE}, drop_last=True)")
            logging.info(f"  Validation batches = {len(val_loader)} (batch size {config.BATCH_SIZE})")
            logging.info(f"  Test batches = {len(test_loader)} (batch size {config.BATCH_SIZE})")
            datasets_created_successfully = True # Mark success

        except Exception as e:
            # Log the specific error but DON'T exit yet
            logging.critical(f"Error creating Datasets or DataLoaders: {e}", exc_info=True)
            # Optionally, re-raise the specific error if you want the script to stop here
            # raise e
    else:
        logging.critical("Skipping Dataset/DataLoader creation because data loading failed.")


    # --- Initialize Model, Loss, Optimizers ---
    model = None # Initialize model to None
    model_initialized_successfully = False
    if datasets_created_successfully: # Only proceed if loaders are ready
        try:
            logging.info("Initializing Model and Loss...")
            model = LateFusionWithTransformer(num_classes=config.NUM_CLASSES,
                                              embed_dim=config.EMBED_DIM,
                                              num_heads=config.NUM_HEADS,
                                              num_layers=config.NUM_LAYERS,
                                              dropout=config.DROPOUT).to(DEVICE)

            total_params = sum(p.numel() for p in model.parameters())
            logging.info(f"Model '{config.MODEL_NAME}' initialized.")
            logging.info(f"  - Total Parameters: {total_params:,}")

            criterion = nn.CrossEntropyLoss()
            logging.info("Loss function (CrossEntropyLoss) initialized.")
            model_initialized_successfully = True # Mark success

        except Exception as e:
            logging.critical(f"Error initializing model or loss function: {e}", exc_info=True)
            # Optionally re-raise
            # raise e
    else:
        logging.critical("Skipping Model initialization because DataLoader creation failed.")


    # --- Training Phase 1: Train Fusion Module ---
    history_phase1 = None # Initialize history
    if model_initialized_successfully: # Only proceed if model is ready
        try:
            logging.info("\n========== Phase 1: Training Fusion Module (Backbones Frozen) ==========")
            model_save_name_phase1 = f"best_model_{config.MODEL_NAME}_phase1.pth"

            # Freeze backbone parameters
            for name, param in model.named_parameters():
                is_backbone = name.startswith('swin.') or name.startswith('efficientnet.')
                param.requires_grad = not is_backbone

            params_to_optimize_p1 = [p for p in model.parameters() if p.requires_grad]
            num_trainable_p1 = sum(p.numel() for p in params_to_optimize_p1)
            logging.info(f"Phase 1 - Optimizing {num_trainable_p1:,} parameters.")
            optimizer_p1 = optim.AdamW(params_to_optimize_p1, lr=config.LEARNING_RATE_PHASE1, weight_decay=0.01)

            # **CRITICAL CHECK**: Ensure train_loader exists before using its length
            if train_loader is None:
                 raise RuntimeError("train_loader was not successfully created. Cannot proceed with Phase 1.")

            # Scheduler for Phase 1 (Cosine Annealing per step)
            total_steps_p1 = len(train_loader) * config.EPOCHS_PHASE1
            scheduler_p1 = optim.lr_scheduler.CosineAnnealingLR(optimizer_p1, T_max=total_steps_p1, eta_min=1e-7)
            logging.info(f"Phase 1 - Cosine LR Scheduler: T_max={total_steps_p1} steps, eta_min={scheduler_p1.eta_min:.1e}")


            model, history_phase1 = train_model(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,
                criterion=criterion,
                optimizer=optimizer_p1,
                scheduler=scheduler_p1,
                num_epochs=config.EPOCHS_PHASE1,
                device=DEVICE,
                patience=config.EARLY_STOPPING_PATIENCE,
                clip_value=config.CLIP_GRAD_NORM,
                phase=1,
                model_save_name=model_save_name_phase1
            )
            # Plot results for phase 1 (only if training completed)
            if history_phase1:
                plot_training_history(history_phase1, 1, config.MODEL_NAME)

        except Exception as e:
            logging.critical(f"Error during Training Phase 1: {e}", exc_info=True)
            # Decide if you want to stop or continue
            # exit()
    else:
         logging.critical("Skipping Phase 1 training because Model initialization failed.")


    # --- Training Phase 2: Fine-tune Entire Model ---
    history_phase2 = None # Initialize history
    # Check if phase 1 finished and model exists
    if model is not None and history_phase1 is not None:
        try:
            logging.info("\n========== Phase 2: Fine-tuning Entire Model ==========")
            model_save_name_phase2 = f"best_model_{config.MODEL_NAME}_phase2_final.pth"

            # Unfreeze all parameters
            for name, param in model.named_parameters():
                param.requires_grad = True

            params_to_optimize_p2 = model.parameters()
            num_trainable_p2 = sum(p.numel() for p in params_to_optimize_p2)
            logging.info(f"Phase 2 - Optimizing {num_trainable_p2:,} parameters.")

            optimizer_p2 = optim.AdamW(params_to_optimize_p2, lr=config.LEARNING_RATE_PHASE2, weight_decay=0.01)

            # **CRITICAL CHECK**: Ensure train_loader exists
            if train_loader is None:
                 raise RuntimeError("train_loader was not successfully created. Cannot proceed with Phase 2.")

            total_steps_p2 = len(train_loader) * config.EPOCHS_PHASE2
            scheduler_p2 = optim.lr_scheduler.CosineAnnealingLR(optimizer_p2, T_max=total_steps_p2, eta_min=1e-8)
            logging.info(f"Phase 2 - Cosine LR Scheduler: T_max={total_steps_p2} steps, eta_min={scheduler_p2.eta_min:.1e}")

            model, history_phase2 = train_model(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,
                criterion=criterion,
                optimizer=optimizer_p2,
                scheduler=scheduler_p2,
                num_epochs=config.EPOCHS_PHASE2,
                device=DEVICE,
                patience=config.EARLY_STOPPING_PATIENCE,
                clip_value=config.CLIP_GRAD_NORM,
                phase=2,
                model_save_name=model_save_name_phase2
            )
            # Plot results for phase 2 (only if training completed)
            if history_phase2:
                plot_training_history(history_phase2, 2, config.MODEL_NAME)

        except Exception as e:
             logging.critical(f"Error during Training Phase 2: {e}", exc_info=True)
             # exit()
    elif model is None:
         logging.critical("Skipping Phase 2 training because Model initialization failed earlier.")
    else: # Model exists but phase 1 failed
         logging.critical("Skipping Phase 2 training because Phase 1 did not complete successfully.")


    # --- Final Evaluation on Test Set ---
    # Check if model and test_loader exist
    if model is not None and test_loader is not None and history_phase2 is not None:
        try:
            logging.info("\n========== Final Evaluation on Test Set ==========")
            model_save_name_phase2 = f"best_model_{config.MODEL_NAME}_phase2_final.pth" # Define again for clarity
            logging.info(f"Loading best fine-tuned model state from: '{os.path.basename(model_save_name_phase2)}'")
            if os.path.exists(model_save_name_phase2):
                try:
                    model.load_state_dict(torch.load(model_save_name_phase2, map_location=DEVICE))
                    logging.info("Best model loaded successfully for final evaluation.")
                except Exception as e:
                     logging.error(f"Failed to load best model '{model_save_name_phase2}'. Evaluating with model state from end of phase 2. Error: {e}")
            else:
                logging.warning(f"Final model file '{model_save_name_phase2}' not found. Evaluating with model state from end of phase 2 training.")


            # Perform evaluation and get attention weights
            test_attention_weights_batches, num_test_samples = evaluate_model(
                model=model,
                test_loader=test_loader,
                device=DEVICE,
                num_classes=config.NUM_CLASSES,
                model_name=config.MODEL_NAME
            )

            # Plot average attention weights from the test set evaluation
            logging.info("\n--- Plotting Average Attention Weights from Test Set ---")
            if test_attention_weights_batches and num_test_samples > 0:
                plot_attention_weights(
                    all_attn_weights_list=test_attention_weights_batches,
                    model_name=config.MODEL_NAME,
                    num_samples_plotted=num_test_samples
                )
            else:
                 logging.warning("Skipping attention weight plotting due to no weights collected or zero samples evaluated.")

        except Exception as e:
             logging.critical(f"Error during final evaluation or attention plotting: {e}", exc_info=True)

    elif model is None:
         logging.critical("Skipping Final Evaluation because Model initialization failed.")
    elif test_loader is None:
         logging.critical("Skipping Final Evaluation because DataLoader creation failed.")
    else: # Phase 2 didn't run
         logging.critical("Skipping Final Evaluation because Phase 2 did not complete successfully.")


    logging.info("========== Experiment Run Finished ==========")

In [ ]:
import json

with open("casia_Late_Transformer_Swin_Efficient.ipynb", "r") as f:
    nb = json.load(f)

if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]

with open("clean_notebook.ipynb", "w") as f:
    json.dump(nb, f)

print("Notebook cleaned!")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import logging
import random
import numpy as np
from thop import profile

# --- Config ---

class Config:
    SEED = 42
    BATCH_SIZE = 8
    IMG_SIZE = 224
    DATA_PATH = "/content/drive/MyDrive/preprocessed_casia_balanced_verified"
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Modèle paramètres
    num_classes = 100  # exemple
    embed_dim = 384
    num_heads = 6
    num_layers = 4
    dropout = 0.1

# --- Seed ---

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# --- Stub fonction de chargement (à adapter selon ton code) ---

def load_or_preprocess_data(data_path, usage=1.0):
    # Ici tu mets ton vrai code de chargement / pré-traitement
    # Retourne (train_img, train_lbl, val_img, val_lbl, test_img, test_lbl)
    # Pour l'exemple, on retourne None
    return None, None, None, None, None, None

# --- Modèle exemple ---

class LateFusionWithTransformer(nn.Module):
    def __init__(self, num_classes, embed_dim, num_heads, num_layers, dropout):
        super().__init__()
        # Modèle fictif avec deux branches de backbone (exemple)
        self.model1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, embed_dim)
        )
        self.model2 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, embed_dim)
        )
        # Fusion + Transformer simulé (simplifié)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dropout=dropout),
            num_layers=num_layers
        )
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x1, x2):
        f1 = self.model1(x1)
        f2 = self.model2(x2)
        fused = torch.stack([f1, f2], dim=0)  # (2, batch, embed_dim)
        out = self.transformer(fused)
        out = out.mean(dim=0)
        return self.classifier(out)

# --- Main ---

if __name__ == "__main__":

    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

    logging.info(f"Démarrage sur {Config.DEVICE}")

    set_seed(Config.SEED)
    print(f"Seed utilisée : {Config.SEED}")

    # Chargement des données (stub)
    try:
        train_img, train_lbl, val_img, val_lbl, test_img, test_lbl = load_or_preprocess_data(Config.DATA_PATH, usage=0.2)
        logging.info("Données chargées avec succès.")
    except Exception as e:
        raise RuntimeError(f"Erreur critique pendant le chargement des données : {e}")

    # Initialisation modèle
    model = LateFusionWithTransformer(
        num_classes=Config.num_classes,
        embed_dim=Config.embed_dim,
        num_heads=Config.num_heads,
        num_layers=Config.num_layers,
        dropout=Config.dropout
    ).to(Config.DEVICE)

    print(f"Nombre total de paramètres : {sum(p.numel() for p in model.parameters())}")

    # Dummy inputs pour profilage
    dummy_input = torch.randn(Config.BATCH_SIZE, 3, Config.IMG_SIZE, Config.IMG_SIZE).to(Config.DEVICE)

    # Calcul FLOPs et params pour chaque branche
    try:
        flops1, params1 = profile(model.model1, inputs=(dummy_input,))
        flops2, params2 = profile(model.model2, inputs=(dummy_input,))

        total_flops = flops1 + flops2
        total_params = params1 + params2

        logging.info(f"FLOPs (branches seules) : {total_flops / 1e9:.3f} GFLOPs")
        logging.info(f"Params (branches seules) : {total_params}")

        print(f"FLOPs (branches seules) : {total_flops / 1e9:.3f} GFLOPs")
        print(f"Params (branches seules) : {total_params}")

    except Exception as e:
        logging.warning(f"Erreur lors du calcul des FLOPs : {e}")

